# 課程 16 - 使用 Microsoft Foundry 部署可擴展代理

在本筆記本中，您將建立一個針對虛構公司 **Contoso** 的 <strong>生產就緒客戶支援代理</strong>。與先前課程不同，重點不在於代理的推理迴圈，而是在其 <em>周圍</em> 所包裝的所有功能，使代理能夠安全地大規模運行：

1. <strong>工具呼叫</strong> — 訂單查詢與工單創建。
2. **RAG** — 從知識庫提供政策答案。
3. <strong>記憶</strong> — 記住跨回合的客戶資訊。
4. <strong>模型路由</strong> — 簡單請求送往小模型，複雜請求送往大型模型。
5. <strong>回應快取</strong> — 針對重複問題無需呼叫模型直接服務。
6. <strong>人工審核</strong> — 超過門檻的退款暫停以待核准。
7. <strong>評估閘道</strong> — 離線測試集阻擋劣質發佈。
8. <strong>可觀察性</strong> — 對每次請求進行 OpenTelemetry 追蹤。

每個部分都是自包含且可執行的。請閱讀每行—這些生產原語故意保持簡潔。


## 設定

在運行此筆記本之前，請確保您已：

1. **擁有一個 Microsoft Foundry 專案** 並已部署聊天室模型（例如 `gpt-5-mini`）。
2. **已使用 Azure CLI 登入** — 在終端機執行 `az login`。
3. **設定必需的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Microsoft Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您已部署模型的名稱。

RAG 區段會在設定 `AZURE_SEARCH_SERVICE_ENDPOINT` 和 `AZURE_SEARCH_API_KEY` 時使用 **Azure AI Search**，否則會退回使用記憶體中搜尋，讓筆記本能在沒有 Search 資源的情況下執行。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. 工具

生產工具針對真實系統執行真實工作。在這裡，我們用純 Python 函數模擬訂單資料庫和票務系統。`@tool` 裝飾器會將它們暴露給代理。

注意 `issue_refund` 對超過門檻的退款使用了 `approval_mode="always_require"` —— 這是我們稍後部署的人機交互原語。


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — 政策知識庫

政策問題（「你的退貨期限是多久？」）應由權威來源回答，而非模型的記憶。我們將一個小型知識庫包裝成搜尋工具。

在生產環境中，這是 **Azure AI Search**；這裡我們提供一個內存關鍵字搜尋，讓筆記本能在任何地方運行，當環境變數存在時會自動切換至 Azure AI Search。


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. 記憶體

一個忘記自己在與誰對話的客服代理，是一個差勁的客服代理。我們會為每位客戶保持一個微型的個人檔案儲存，並將一段簡短的摘要注入代理的指示中。在生產環境中這是一個記憶體服務（見第13課）；在這裡以字典展示此模式。 


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. 模型路由和回應快取

兩個成本槓桿接入單一請求處理器：

- <strong>路由</strong>：一個廉價的啟發式分類器決定請求是否需要小型模型或大型模型。
- <strong>快取</strong>：標準化的重複問題直接從快取中提供，無需呼叫模型。

此處的分類器刻意保持簡單。投入生產時你會用流量進行驗證，並可用 Foundry 的模型路由器替代它。


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. 代理、人類審批與可觀察性

現在我們從上述工具組裝代理，並將每個請求包裝在 OpenTelemetry 的 span 中。`handle_support_request` 函數是生產環境下的請求處理器：緩存 → 路由 → 追蹤 → 執行 → 緩存。

人類審批由框架處理：因為 `issue_refund` 的 `approval_mode="always_require"`，執行會暫停並顯示需要審批的請求，我們會明確地解決該審批。


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. 評估關卡

這是課程中的發佈關卡：一個離線測試集為代理進行評分，只有通過率達到門檻時才會繼續部署。這裡的評分器是一個簡單的關鍵詞重疊檢查，以保持筆記本自包含；在生產環境中，你會使用大型語言模型作為評判者或框架評估器（參見第10課）。


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## 組合起來：模擬發布

下方的儲存格展示了本課程描述的整個迴圈：執行評估關卡，只有通過才會「部署」。這是你在 CI 中推廣代理版本到 Foundry Agent Service 前會執行的模式。


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## 摘要

你組裝了一個生產就緒的客戶支援代理，集成了所有運營相關問題：

- **工具、RAG 和記憶體** 賦予代理能力和背景資訊。
- <strong>模型路由和快取</strong> 控制延遲和成本。
- <strong>人工審核</strong> 防範高風險操作，例如大額退款。
- <strong>評估門檻</strong> 阻止問題版本發佈。
- <strong>追蹤</strong> 使每個請求都可被觀察。

### 挑戰

擴展這個代理以：

1. <strong>支援多模型</strong> — 添加第三個「推理」層級並將升級/投訴導向該層。
2. <strong>加入評估門檻</strong> — 擴展 `TEST_CASES` 包含退款批准場景，並確認門檻能捕捉迴歸。
3. <strong>新增成本感知路由</strong> — 追蹤每個請求的預估成本（小額、大額與快取）並在混合查詢批次後打印成本報告。

在下一課，你將進行相反的旅程，使用 Microsoft Foundry Local 與 Qwen，完全在你自己的機器上執行代理。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
